# MQTT Publish 應用程式

本課程將學習如何使用 paho-mqtt 套件來發布（publish）訊息到 MQTT Broker。

## MQTT 基本概念
- **Publisher（發布者）**：發送訊息的客戶端
- **Broker（代理伺服器）**：接收和分發訊息的伺服器
- **Topic（主題）**：訊息的主題路徑，例如：`sensor/temperature`
- **Message（訊息）**：要發送的內容


In [ ]:
# 導入 paho-mqtt 套件
import paho.mqtt.client as mqtt
import time

print("paho-mqtt 套件已成功導入！")


## 基礎版：簡單的 MQTT Publish 程式


In [ ]:
# MQTT Broker 設定
BROKER_HOST = "localhost"  # 或使用 "test.mosquitto.org" 作為公開測試伺服器
BROKER_PORT = 1883         # MQTT 預設端口
TOPIC = "test/temperature" # 要發布的主題

# 建立 MQTT 客戶端
client = mqtt.Client()

# 連接到 MQTT Broker
print(f"正在連接到 {BROKER_HOST}:{BROKER_PORT}...")
client.connect(BROKER_HOST, BROKER_PORT, 60)

# 發布訊息
message = "25.5"  # 要發送的訊息內容
client.publish(TOPIC, message)
print(f"已發布訊息到主題 '{TOPIC}': {message}")

# 斷開連接
client.disconnect()
print("已斷開連接")


## 進階版：包含連接狀態回調和持續發布


In [ ]:
# MQTT Broker 設定
BROKER_HOST = "localhost"  # 修改為您的 MQTT Broker 地址
BROKER_PORT = 1883
TOPIC = "sensor/temperature"

# 連接狀態回調函數
def on_connect(client, userdata, flags, rc):
    """當連接到 Broker 時被調用"""
    if rc == 0:
        print("✓ 成功連接到 MQTT Broker")
    else:
        print(f"✗ 連接失敗，錯誤代碼：{rc}")

def on_publish(client, userdata, mid):
    """當訊息發布成功時被調用"""
    print(f"✓ 訊息已成功發布 (mid: {mid})")

# 建立 MQTT 客戶端
client = mqtt.Client()

# 設定回調函數
client.on_connect = on_connect
client.on_publish = on_publish

# 連接到 Broker
print(f"正在連接到 {BROKER_HOST}:{BROKER_PORT}...")
client.connect(BROKER_HOST, BROKER_PORT, 60)

# 啟動網路循環（處理網路流量）
client.loop_start()

# 等待連接建立
time.sleep(1)

# 發布多筆訊息
for i in range(5):
    temperature = 20 + i * 0.5  # 模擬溫度數據
    message = str(temperature)
    
    # 發布訊息，qos=1 表示至少傳送一次
    result = client.publish(TOPIC, message, qos=1)
    
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"發布訊息 {i+1}: {message}°C 到主題 '{TOPIC}'")
    else:
        print(f"發布失敗，錯誤代碼：{result.rc}")
    
    time.sleep(1)  # 等待 1 秒

# 停止網路循環並斷開連接
client.loop_stop()
client.disconnect()
print("已斷開連接")


## 測試版：快速測試 MQTT Publish（使用公開測試伺服器）


In [ ]:
# ============================================
# MQTT Publish 測試程式
# 使用公開測試伺服器，無需安裝本地 Broker
# ============================================

# MQTT 設定（使用公開測試伺服器）
BROKER_HOST = "test.mosquitto.org"  # 公開測試伺服器，無需認證
BROKER_PORT = 1883
TOPIC = "test/python/mqtt"  # 測試主題（可自行修改）

# 連接狀態回調
def on_connect_test(client, userdata, flags, rc):
    if rc == 0:
        print("✓ 成功連接到測試伺服器！")
    else:
        print(f"✗ 連接失敗，錯誤代碼：{rc}")

def on_publish_test(client, userdata, mid):
    print(f"✓ 訊息發布成功 (訊息ID: {mid})")

# 建立客戶端
client = mqtt.Client()
client.on_connect = on_connect_test
client.on_publish = on_publish_test

# 連接到測試伺服器
print("=" * 50)
print(f"正在連接到測試伺服器: {BROKER_HOST}:{BROKER_PORT}")
print(f"測試主題: {TOPIC}")
print("=" * 50)

try:
    client.connect(BROKER_HOST, BROKER_PORT, 60)
    client.loop_start()
    time.sleep(2)  # 等待連接建立
    
    # 測試 1：發布簡單文字訊息
    print("\n【測試 1】發布簡單文字訊息")
    test_message = "Hello MQTT! 這是測試訊息"
    result = client.publish(TOPIC, test_message, qos=1)
    if result.rc == mqtt.MQTT_ERR_SUCCESS:
        print(f"  → 已發布: {test_message}")
    time.sleep(1)
    
    # 測試 2：發布數字訊息
    print("\n【測試 2】發布數字訊息")
    number_message = "42"
    client.publish(TOPIC, number_message, qos=1)
    print(f"  → 已發布: {number_message}")
    time.sleep(1)
    
    # 測試 3：發布多筆訊息
    print("\n【測試 3】連續發布多筆訊息")
    for i in range(3):
        msg = f"測試訊息 #{i+1} - 時間: {time.strftime('%H:%M:%S')}"
        client.publish(TOPIC, msg, qos=1)
        print(f"  → 已發布: {msg}")
        time.sleep(1)
    
    print("\n" + "=" * 50)
    print("✓ 所有測試完成！")
    print("=" * 50)
    
except Exception as e:
    print(f"✗ 發生錯誤: {e}")
finally:
    # 清理
    client.loop_stop()
    client.disconnect()
    print("\n已斷開連接")


## 實用範例：發布感測器數據（JSON 格式）


In [ ]:
import json
import random

# MQTT 設定
BROKER_HOST = "test.mosquitto.org"  # 或改為 "localhost"
BROKER_PORT = 1883
TOPIC = "test/sensor/data"

# 建立客戶端並設定回調
client = mqtt.Client()
client.on_connect = on_connect_test
client.on_publish = on_publish_test

# 連接並啟動
client.connect(BROKER_HOST, BROKER_PORT, 60)
client.loop_start()
time.sleep(2)

# 模擬感測器數據並發布
print("開始發布感測器數據...")
for i in range(3):
    # 建立感測器數據字典
    sensor_data = {
        "sensor_id": "sensor_001",
        "temperature": round(20 + random.uniform(-2, 2), 2),
        "humidity": round(50 + random.uniform(-5, 5), 2),
        "timestamp": time.time()
    }
    
    # 將字典轉換為 JSON 字串
    message = json.dumps(sensor_data, ensure_ascii=False)
    
    # 發布 JSON 格式的訊息
    client.publish(TOPIC, message, qos=1)
    print(f"發布感測器數據 {i+1}: {message}")
    
    time.sleep(2)

# 清理
client.loop_stop()
client.disconnect()
print("完成！")


## 注意事項與使用說明

### 測試伺服器選擇

1. **公開測試伺服器**（推薦用於測試）：
   - `test.mosquitto.org` - 無需認證，可直接使用
   - 適合快速測試和學習

2. **本地伺服器**（需要先安裝）：
   - `localhost` 或 `127.0.0.1`
   - 需要先安裝並啟動 Mosquitto MQTT Broker
   - 安裝方式：`sudo apt-get install mosquitto mosquitto-clients`

### QoS 等級說明

- **qos=0**：最多傳送一次（最快，可能遺失訊息）
- **qos=1**：至少傳送一次（保證送達，可能重複）
- **qos=2**：只傳送一次（保證送達且不重複，最慢）

### 認證設定

如果使用需要認證的 Broker，在連接前加入：
```python
client.username_pw_set("username", "password")
```

### 驗證訊息是否發布成功

可以使用 MQTT 客戶端工具訂閱相同主題來驗證：
```bash
# 使用 mosquitto_sub 訂閱（需要安裝 mosquitto-clients）
mosquitto_sub -h test.mosquitto.org -t "test/python/mqtt"
```

或使用線上工具：
- https://www.hivemq.com/mqtt-toolbox/
- https://mqttx.app/
